# ĐỒ ÁN MÔN HỌC: XỬ LÝ ẢNH — NHẬN DIỆN CHẤT LƯỢNG GẠO
## Phân hệ 3: Trích xuất đặc trưng hạt gạo (Feature Extraction) — Tuân Đoàn

Notebook này thực hiện trích xuất các đặc trưng định lượng cho từng vùng hạt gạo đã được tách phân đoạn từ phân hệ trước, kết hợp đặc trưng **Hình dạng**, **Màu sắc** và **Kết cấu** bằng phương pháp xử lý ảnh thuần túy[cite: 4, 6].

**Đặc tả kỹ thuật:**
* **Đặc trưng về hình dạng (`Shape`):** Tính diện tích (`cv2.contourArea`), chu vi (`cv2.arcLength`), chiều dài/rộng và tỷ lệ khung bao xoay tối thiểu (`cv2.minAreaRect`), độ tròn (`Circularity`) và độ đặc (`Solidity`) từ mask của từng hạt[cite: 4].
* **Đặc trưng về màu sắc (`Color`):** Tính giá trị `RGB` trung bình bên trong vùng hạt, chuyển đổi sang không gian màu `HSV` để đo độ sáng (`V`) và độ bão hòa (`S`), xác định tỷ lệ pixel vàng hoặc sẫm màu và xây dựng vector đặc trưng màu cho từng hạt[cite: 4].
* **Đặc trưng về kết cấu (`Texture - GLCM`):** Lượng tử hóa mức xám vùng hạt và tính ma trận đồng xuất hiện mức xám (`GLCM`) với các chỉ số `Contrast`, `Homogeneity`, `Energy` và `Correlation`[cite: 4].
* **Tổng hợp & kiểm tra dữ liệu:** Hợp nhất `Shape + Color + Texture` thành vector đặc trưng hoàn chỉnh cho từng hạt, kiểm tra dữ liệu thiếu/bất thường (`NaN`, `Inf`) và xuất bảng đặc trưng phục vụ phân loại ở phân hệ tiếp theo[cite: 4].

In [ ]:
# %% [code]
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN TƯƠNG ĐỐI
# ==============================================================================
try:
    PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
except NameError:
    PROJECT_ROOT = os.getcwd()

# Ảnh màu đã tiền xử lý từ TV1 (dùng cho trích xuất màu sắc)
BATCH_COLOR_DIR = os.path.join(
    PROJECT_ROOT, "processed_dataset", "2_batch_rice_images", "color_clean"
)

# File CSV và dữ liệu từ TV2 (dùng cho hình dạng, kết cấu & liên kết vùng hạt)
SEG_INDEX_CSV = os.path.join(
    PROJECT_ROOT, "segmented_dataset", "segmented_grains.csv"
)

# Đầu ra của TV3
FEAT_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "features_dataset")
FEAT_CSV_PATH = os.path.join(FEAT_OUTPUT_DIR, "grain_features.csv")
FEAT_CLEAN_CSV_PATH = os.path.join(FEAT_OUTPUT_DIR, "grain_features_clean.csv")

# ==============================================================================
# 2. CẤU HÌNH THAM SỐ TRÍCH XUẤT ĐẶC TRƯNG
# ==============================================================================
FEAT_CONFIG = {
    # Ngưỡng xác định pixel vàng / sẫm màu trong HSV (OpenCV: H [0..179], S,V [0..255])
    "yellow_h_min": 15,
    "yellow_h_max": 38,
    "yellow_s_min": 40,
    "yellow_v_min": 60,
    "dark_v_max": 85,
    "dark_s_min": 30,
    # Cấu hình GLCM (kết cấu)
    "glcm_levels": 64,
    "glcm_distances": [1, 2],
    "glcm_angles_deg": [0, 45, 90, 135],
}

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Nguồn ảnh màu TV1: {BATCH_COLOR_DIR}")
print(f"Nguồn CSV TV2: {SEG_INDEX_CSV}")

In [11]:
# %% [code]
def extract_shape_features_from_tv2(row: pd.Series, roi_mask: np.ndarray | None) -> dict:
    """
    Trích xuất đặc trưng hình dạng:
    - Diện tích (cv2.contourArea / ưu tiên diện tích hợp lệ từ TV2 nếu có)
    - Chu vi (cv2.arcLength)
    - Chiều dài, chiều rộng, tỷ lệ dài/rộng (cv2.minAreaRect)
    - Độ tròn (Circularity)
    - Độ đặc (Solidity)
    """
    area_from_csv = row.get("area_px", np.nan)

    if roi_mask is None or roi_mask.size == 0:
        w_csv = float(row.get("bbox_w", np.nan))
        h_csv = float(row.get("bbox_h", np.nan))
        length_csv = max(w_csv, h_csv) if (not pd.isna(w_csv) and not pd.isna(h_csv)) else np.nan
        width_csv = min(w_csv, h_csv) if (not pd.isna(w_csv) and not pd.isna(h_csv)) else np.nan
        aspect_csv = (length_csv / width_csv) if (width_csv and width_csv > 1e-6) else np.nan
        return {
            "shape_area": float(area_from_csv) if not pd.isna(area_from_csv) else np.nan,
            "shape_perimeter": np.nan,
            "shape_length": length_csv,
            "shape_width": width_csv,
            "shape_aspect_ratio": aspect_csv,
            "shape_circularity": np.nan,
            "shape_solidity": np.nan,
        }

    binary = (roi_mask > 0).astype(np.uint8)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return {
            "shape_area": float(area_from_csv) if not pd.isna(area_from_csv) else np.nan,
            "shape_perimeter": np.nan,
            "shape_length": np.nan,
            "shape_width": np.nan,
            "shape_aspect_ratio": np.nan,
            "shape_circularity": np.nan,
            "shape_solidity": np.nan,
        }

    cnt = max(contours, key=cv2.contourArea)
    cnt_area = float(cv2.contourArea(cnt))
    area = float(area_from_csv) if (not pd.isna(area_from_csv) and float(area_from_csv) > 0) else cnt_area
    perimeter = float(cv2.arcLength(cnt, True))

    (_, _), (rect_w, rect_h), _ = cv2.minAreaRect(cnt)
    length = float(max(rect_w, rect_h))
    width = float(min(rect_w, rect_h))
    aspect_ratio = float(length / width) if width > 1e-6 else np.nan

    circularity = (
        float((4.0 * np.pi * cnt_area) / (perimeter * perimeter))
        if perimeter > 1e-6
        else np.nan
    )

    hull = cv2.convexHull(cnt)
    hull_area = float(cv2.contourArea(hull))
    solidity = float(cnt_area / hull_area) if hull_area > 1e-6 else np.nan

    return {
        "shape_area": round(area, 4),
        "shape_perimeter": round(perimeter, 4),
        "shape_length": round(length, 4),
        "shape_width": round(width, 4),
        "shape_aspect_ratio": round(aspect_ratio, 4) if not np.isnan(aspect_ratio) else np.nan,
        "shape_circularity": round(circularity, 4) if not np.isnan(circularity) else np.nan,
        "shape_solidity": round(solidity, 4) if not np.isnan(solidity) else np.nan,
    }


def extract_texture_glcm_features(
    gray_patch: np.ndarray | None,
    mask_patch: np.ndarray | None,
    cfg: dict = FEAT_CONFIG,
) -> dict:
    """
    Trích xuất đặc trưng kết cấu GLCM (Contrast, Homogeneity, Energy, Correlation)
    trên vùng hạt.
    """
    if gray_patch is None or mask_patch is None or gray_patch.size == 0 or mask_patch.size == 0:
        return {
            "glcm_contrast": np.nan,
            "glcm_homogeneity": np.nan,
            "glcm_energy": np.nan,
            "glcm_correlation": np.nan,
        }

    mask_bool = mask_patch > 0
    if np.count_nonzero(mask_bool) < 4:
        return {
            "glcm_contrast": np.nan,
            "glcm_homogeneity": np.nan,
            "glcm_energy": np.nan,
            "glcm_correlation": np.nan,
        }

    gray_2d = gray_patch if gray_patch.ndim == 2 else cv2.cvtColor(gray_patch, cv2.COLOR_BGR2GRAY)
    mean_val = float(np.mean(gray_2d[mask_bool]))

    clean_gray = np.where(mask_bool, gray_2d.astype(np.float64), mean_val)
    clean_gray = np.clip(clean_gray, 0, 255).astype(np.uint8)

    levels = int(cfg.get("glcm_levels", 64))
    quantized = np.floor(clean_gray.astype(np.float32) * (levels / 256.0)).astype(np.uint8)
    quantized = np.clip(quantized, 0, levels - 1)

    angles_rad = [np.deg2rad(a) for a in cfg.get("glcm_angles_deg", [0, 45, 90, 135])]
    distances = cfg.get("glcm_distances", [1, 2])

    glcm = graycomatrix(
        quantized,
        distances=distances,
        angles=angles_rad,
        levels=levels,
        symmetric=True,
        normed=True,
    )

    return {
        "glcm_contrast": round(float(np.mean(graycoprops(glcm, "contrast"))), 6),
        "glcm_homogeneity": round(float(np.mean(graycoprops(glcm, "homogeneity"))), 6),
        "glcm_energy": round(float(np.mean(graycoprops(glcm, "energy"))), 6),
        "glcm_correlation": round(float(np.mean(graycoprops(glcm, "correlation"))), 6),
    }

In [12]:
# %% [code]
def load_tv1_color_image(batch_id: str, color_cache: dict) -> np.ndarray | None:
    """
    Nạp ảnh màu đã tiền xử lý từ TV1 (processed_dataset/2_batch_rice_images/color_clean/).
    """
    if batch_id in color_cache:
        return color_cache[batch_id]

    if not os.path.exists(BATCH_COLOR_DIR):
        color_cache[batch_id] = None
        return None

    candidate = os.path.join(BATCH_COLOR_DIR, f"{batch_id}.jpg")
    img_path = candidate if os.path.exists(candidate) else None

    if img_path is None:
        for fname in sorted(os.listdir(BATCH_COLOR_DIR)):
            if os.path.splitext(fname)[0] == batch_id:
                img_path = os.path.join(BATCH_COLOR_DIR, fname)
                break

    img = cv2.imread(img_path) if img_path and os.path.exists(img_path) else None
    color_cache[batch_id] = img
    return img


def get_color_region_for_grain(
    row: pd.Series,
    img_color_tv1: np.ndarray | None,
    target_shape: tuple[int, int],
    label_map_cache: dict,
) -> tuple[np.ndarray | None, np.ndarray]:
    """
    Lấy vùng ảnh màu của hạt từ ảnh màu TV1 theo thông tin trong CSV của TV2
    (ưu tiên mask chính xác từ label_map của TV2, dự phòng bounding box từ CSV).
    """
    h_t, w_t = target_shape
    default_mask = np.ones((h_t, w_t), dtype=np.uint8) * 255

    if img_color_tv1 is None or img_color_tv1.size == 0:
        return None, default_mask

    h_img, w_img = img_color_tv1.shape[:2]

    # Nếu TV2 có label_map_path và grain_id, lấy mask chuẩn của đúng hạt đó
    lmap_rel = row.get("label_map_path", "")
    gid = row.get("grain_id", None)
    grain_mask_full = None

    if lmap_rel and not pd.isna(lmap_rel) and gid is not None and not pd.isna(gid):
        lmap_path = str(lmap_rel).strip()
        lmap_path = lmap_path if os.path.isabs(lmap_path) else os.path.join(PROJECT_ROOT, lmap_path)
        if os.path.exists(lmap_path):
            if lmap_path not in label_map_cache:
                label_map_cache[lmap_path] = np.load(lmap_path)
            lmap = label_map_cache[lmap_path]
            if lmap.shape[:2] == (h_img, w_img):
                grain_mask_full = (lmap == int(gid)).astype(np.uint8) * 255

    x = int(row["bbox_x"])
    y = int(row["bbox_y"])
    w = int(row["bbox_w"])
    h = int(row["bbox_h"])

    pad_y = max(0, (h_t - h) // 2)
    pad_x = max(0, (w_t - w) // 2)

    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(w_img, x1 + w_t)
    y2 = min(h_img, y1 + h_t)

    color_patch = img_color_tv1[y1:y2, x1:x2]
    if color_patch.size == 0:
        return None, default_mask

    if color_patch.shape[:2] != (h_t, w_t):
        color_patch = cv2.resize(color_patch, (w_t, h_t), interpolation=cv2.INTER_NEAREST)

    if grain_mask_full is not None:
        mask_patch = grain_mask_full[y1:y2, x1:x2]
        if mask_patch.shape[:2] != (h_t, w_t):
            mask_patch = cv2.resize(mask_patch, (w_t, h_t), interpolation=cv2.INTER_NEAREST)
        if np.count_nonzero(mask_patch) > 0:
            return color_patch, mask_patch

    return color_patch, default_mask


def extract_color_features_from_tv1(
    color_patch: np.ndarray | None,
    mask_patch: np.ndarray,
    cfg: dict = FEAT_CONFIG,
) -> tuple[dict, np.ndarray]:
    """
    Trích xuất đặc trưng màu từ ảnh màu của TV1:
    - Giá trị RGB trung bình bên trong vùng hạt
    - Chuyển sang HSV, tính độ sáng (V) và độ bão hòa (S)
    - Tỷ lệ pixel vàng hoặc sẫm màu
    - Vector đặc trưng màu chuẩn hóa
    """
    mask_bool = mask_patch > 0
    num_px = int(np.count_nonzero(mask_bool))

    if color_patch is None or color_patch.size == 0 or num_px == 0:
        empty_feats = {
            "color_r_mean": np.nan,
            "color_g_mean": np.nan,
            "color_b_mean": np.nan,
            "color_h_mean": np.nan,
            "color_s_mean": np.nan,
            "color_v_mean": np.nan,
            "color_s_std": np.nan,
            "color_v_std": np.nan,
            "color_yellow_ratio": np.nan,
            "color_dark_ratio": np.nan,
        }
        return empty_feats, np.full(8, np.nan, dtype=np.float64)

    rgb = cv2.cvtColor(color_patch, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(color_patch, cv2.COLOR_BGR2HSV)

    r_vals = rgb[:, :, 0][mask_bool].astype(np.float64)
    g_vals = rgb[:, :, 1][mask_bool].astype(np.float64)
    b_vals = rgb[:, :, 2][mask_bool].astype(np.float64)

    h_vals = hsv[:, :, 0][mask_bool].astype(np.float64)
    s_vals = hsv[:, :, 1][mask_bool].astype(np.float64)
    v_vals = hsv[:, :, 2][mask_bool].astype(np.float64)

    r_mean, g_mean, b_mean = float(np.mean(r_vals)), float(np.mean(g_vals)), float(np.mean(b_vals))
    h_mean, s_mean, v_mean = float(np.mean(h_vals)), float(np.mean(s_vals)), float(np.mean(v_vals))
    s_std, v_std = float(np.std(s_vals)), float(np.std(v_vals))

    yellow_mask = (
        (h_vals >= cfg["yellow_h_min"])
        & (h_vals <= cfg["yellow_h_max"])
        & (s_vals >= cfg["yellow_s_min"])
        & (v_vals >= cfg["yellow_v_min"])
    )
    dark_mask = (v_vals <= cfg["dark_v_max"]) | ((v_vals <= 110) & (s_vals >= cfg["dark_s_min"]))

    yellow_ratio = float(np.count_nonzero(yellow_mask)) / float(num_px)
    dark_ratio = float(np.count_nonzero(dark_mask)) / float(num_px)

    color_feats = {
        "color_r_mean": round(r_mean, 4),
        "color_g_mean": round(g_mean, 4),
        "color_b_mean": round(b_mean, 4),
        "color_h_mean": round(h_mean, 4),
        "color_s_mean": round(s_mean, 4),
        "color_v_mean": round(v_mean, 4),
        "color_s_std": round(s_std, 4),
        "color_v_std": round(v_std, 4),
        "color_yellow_ratio": round(yellow_ratio, 6),
        "color_dark_ratio": round(dark_ratio, 6),
    }

    color_vec = np.array([
        r_mean / 255.0,
        g_mean / 255.0,
        b_mean / 255.0,
        h_mean / 179.0,
        s_mean / 255.0,
        v_mean / 255.0,
        yellow_ratio,
        dark_ratio,
    ], dtype=np.float64)

    return color_feats, color_vec

In [ ]:
# %% [code]
def _read_optional_gray(rel_path_val) -> np.ndarray | None:
    if not rel_path_val or pd.isna(rel_path_val):
        return None
    p = str(rel_path_val).strip()
    full_p = p if os.path.isabs(p) else os.path.join(PROJECT_ROOT, p)
    if not os.path.exists(full_p):
        return None
    return cv2.imread(full_p, cv2.IMREAD_GRAYSCALE)


def process_grain_row(
    row: pd.Series,
    color_cache: dict,
    label_map_cache: dict,
    cfg: dict = FEAT_CONFIG,
) -> dict:
    roi_mask = _read_optional_gray(row.get("roi_mask_path", ""))
    roi_gray = _read_optional_gray(row.get("roi_gray_path", ""))

    if roi_mask is not None:
        target_shape = roi_mask.shape[:2]
    else:
        target_shape = (max(1, int(row.get("bbox_h", 16))), max(1, int(row.get("bbox_w", 16))))

    # 1. Màu sắc lấy từ ảnh màu đã tiền xử lý của TV1
    img_color_tv1 = load_tv1_color_image(str(row["batch_id"]), color_cache)
    color_patch, color_mask_patch = get_color_region_for_grain(
        row=row,
        img_color_tv1=img_color_tv1,
        target_shape=target_shape,
        label_map_cache=label_map_cache,
    )
    color_feats, color_vec = extract_color_features_from_tv1(color_patch, color_mask_patch, cfg=cfg)

    # 2. Hình dạng & kết cấu lấy từ thông tin CSV / vùng hạt của TV2
    active_mask = roi_mask if roi_mask is not None else color_mask_patch
    shape_feats = extract_shape_features_from_tv2(row=row, roi_mask=active_mask)

    gray_for_glcm = roi_gray
    if gray_for_glcm is None and color_patch is not None:
        gray_for_glcm = cv2.cvtColor(color_patch, cv2.COLOR_BGR2GRAY)
    texture_feats = extract_texture_glcm_features(
        gray_patch=gray_for_glcm,
        mask_patch=active_mask,
        cfg=cfg,
    )

    # 3. Tổng hợp vector đặc trưng (Shape + Color + Texture)
    combined_vec = np.concatenate([
        np.array([
            shape_feats["shape_area"],
            shape_feats["shape_perimeter"],
            shape_feats["shape_length"],
            shape_feats["shape_width"],
            shape_feats["shape_aspect_ratio"],
            shape_feats["shape_circularity"],
            shape_feats["shape_solidity"],
        ], dtype=np.float64),
        color_vec,
        np.array([
            texture_feats["glcm_contrast"],
            texture_feats["glcm_homogeneity"],
            texture_feats["glcm_energy"],
            texture_feats["glcm_correlation"],
        ], dtype=np.float64),
    ])

    is_abnormal = bool(np.isnan(combined_vec).any() or np.isinf(combined_vec).any())

    out = {
        "grain_uid": row.get("grain_uid", ""),
        "batch_id": row.get("batch_id", ""),
        "grain_id": int(row["grain_id"]) if not pd.isna(row.get("grain_id", np.nan)) else -1,
        "centroid_x": row.get("centroid_x", np.nan),
        "centroid_y": row.get("centroid_y", np.nan),
    }
    out.update(shape_feats)
    out.update(color_feats)
    out.update(texture_feats)
    out["color_vector"] = np.round(color_vec, 6).tolist()
    out["feature_vector"] = np.round(combined_vec, 6).tolist()
    out["feature_dim"] = int(len(combined_vec))
    out["feature_status"] = "abnormal_or_missing" if is_abnormal else "ok"
    return out


def run_tv3_feature_pipeline(
    seg_csv_path: str = SEG_INDEX_CSV,
    cfg: dict = FEAT_CONFIG,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not os.path.exists(seg_csv_path):
        raise FileNotFoundError(f"Không tìm thấy CSV của TV2: {seg_csv_path}")

    df_tv2 = pd.read_csv(seg_csv_path)
    if df_tv2.empty:
        raise ValueError("File segmented_grains.csv của TV2 không có dữ liệu.")

    os.makedirs(FEAT_OUTPUT_DIR, exist_ok=True)
    color_cache = {}
    label_map_cache = {}

    records = [
        process_grain_row(row, color_cache=color_cache, label_map_cache=label_map_cache, cfg=cfg)
        for _, row in df_tv2.iterrows()
    ]

    df_all = pd.DataFrame(records)
    df_clean = df_all[df_all["feature_status"] == "ok"].reset_index(drop=True)

    num_cols = df_all.select_dtypes(include=[np.number]).columns.tolist()
    nan_cols = int(df_all[num_cols].isna().any().sum())
    inf_cols = int(np.isinf(df_all[num_cols].to_numpy(dtype=np.float64)).any(axis=0).sum())

    df_all.to_csv(FEAT_CSV_PATH, index=False, encoding="utf-8-sig")
    df_clean.to_csv(FEAT_CLEAN_CSV_PATH, index=False, encoding="utf-8-sig")

    print("=" * 65)
    print("KẾT QUẢ TRÍCH XUẤT ĐẶC TRƯNG TV3:")
    print(f"  - Tổng số hạt từ CSV TV2 : {len(df_all)}")
    print(f"  - Số hạt hợp lệ (ok)     : {len(df_clean)}")
    print(f"  - Số hạt bất thường/thiếu: {len(df_all) - len(df_clean)}")
    print(f"  - Cột có NaN / Inf       : {nan_cols} / {inf_cols}")
    print(f"  - File đầy đủ            : {FEAT_CSV_PATH}")
    print(f"  - File sạch              : {FEAT_CLEAN_CSV_PATH}")
    print("=" * 65)

    return df_all, df_clean


df_features_all, df_features_clean = run_tv3_feature_pipeline()
df_features_clean.head(10)